<a href="https://colab.research.google.com/github/enyachang1119-debug/DS2002/blob/main/Copy_of_2026_10_02_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _one line item in an order_

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [10]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total

# Flatten the nested orders into one row per line item.
lines = pd.json_normalize(
    orders_json,
    record_path='lines', # Extract
    meta=['order', 'vendor_id'] # Keep the order and vendor IDs on every row
)

# Check that the flattened table contains exactly 4 line items
assert len(lines) == 4

# Check that the total quantity across all line items is 7
assert lines['qty'].sum() == 7

lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-01


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [11]:
# TODO
vendors = pd.json_normalize(vendors_json) # Convert the vendor JSON into a DataFrame

# Add vendor information to each line item
joined = lines.merge(
    vendors,
    on='vendor_id', # Match rows using the vendor I
    how='left', # Keep every line item, including those without a match.
    indicator=True # Add a _merge column to show whether each row matched
)
joined

unmatched = joined[joined['_merge'] == 'left_only'] # Select line items that have no matching vendor

unmatched_units = unmatched.groupby('vendor_id')['qty'].sum() # Calculate the total quantity for each unmatched vendor

# Report unmatched vendor IDs and their quantities
print(unmatched_units)
# Report the total quantity across all unmatched line items
print('Total unmatched units:', unmatched['qty'].sum())

#All vendor IDs in the provided orders match the vendor list so there are no unmatched vendors and the unmatched quantity is 0

Series([], Name: qty, dtype: int64)
Total unmatched units: 0


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [12]:
# TODO
joined['name'] = joined['name'].fillna('Unknown vendor') # Label missing vendor names as 'Unknown vendor' so their quantities remain included in the totals instead of being dropped by groupby

vendor_totals = joined.groupby('name')['qty'].sum() # Group line items by vendor name and sum the quantities for each vendor

vendor_totals # Display the total quantity per vendor

,qty
name,
Cav Merch,1
Hoos Burgers,6


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [14]:
# TODO
# Group line items by zone and sum their quantities
zone_totals = joined.groupby('zone')['qty'].sum()
print(zone_totals) # Display the total quantity for each zone
print('Zone with the most items:', zone_totals.idxmax()) # Find the zone with the highest total quantity
print('Total units counted by zone:', zone_totals.sum()) #Report the total quantity included in the zone groups

# Sum quantities for rows with missing zones to account for any units excluded from the zone totals
missing_zone_units = joined.loc[
    joined['zone'].isna(), 'qty'
].sum()

print('Units without a zone:', missing_zone_units)

# Zone A sold the most items (7 units)
# The zone totals include all 7 units, so the difference is 0 and no units were excluded because every line item has a zone

zone
A    7
Name: qty, dtype: int64
Zone with the most items: A
Total units counted by zone: 7
Units without a zone: 0


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [16]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
# Keep orders that contain a 'lines' key to avoid an error during flattening
# Order 5 produces no line-item rows because it has no 'lines' key
ragged_orders = [
    order for order in ragged_json
    if 'lines' in order
]

# Flatten the available line items and keep their order and vendor IDs
# A line item without a 'qty' field receives a missing value
ragged_lines = pd.json_normalize(
    ragged_orders,
    record_path='lines',
    meta=['order', 'vendor_id']
)

# Use pandas' nullable integer type to preserve missing quantities as <NA>
# A missing quantity is unknown, while zero is a known quantity of 0
ragged_lines['qty'] = ragged_lines['qty'].astype('Int64')
ragged_lines # Display the result: Soda has quantity 2, and Fries has quantity <NA>

,item,qty,order,vendor_id
0,Soda,2,4,V-01
1,Fries,<NA>,6,V-01


### Q6 — Validate

In [17]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

_a -One row per line item is a useful shape because it gives each item its own quantity and keep its own order and vendor IDs, which makes it easy to join vendor information and calculate the total quantities by item, vendor, or zone. A question becomes easy is something like "How many units of each item were sold" because it gets easier to group and find sum._


_b - The practical difference between a missing quantity and a quantity of zero is that a missing quantity means that quantitiy is unknown or was not provided, while a quantity of zero means the quantity is known to be 0. If we replace missing quantities with zere, it might destort averages and underestimate the total._